# Dashboard de Monitoramento — Camada Gold
**Tech Challenge Fase 2 | Observabilidade da Pipeline**

Monitoramento operacional dos produtos analíticos:
- Volume e cobertura por tabela Gold
- Distribuição de indicadores
- Alertas de qualidade
- KPIs executivos

In [0]:
# =============================================================================
# CARREGAR DADOS DA PIPELINE PRINCIPAL
# =============================================================================
%run "./Pipeline Alfabetizacao Brasil Tech Challenge"

from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"📊 Dashboard gerado em: {RUN_TIMESTAMP}")

In [0]:
# =============================================================================
# KPI 1 — VOLUMETRIA POR CAMADA
# =============================================================================

print("\n" + "="*70)
print(f"  MONITORAMENTO DA PIPELINE — {RUN_TIMESTAMP}")
print("="*70)

print("\n🟥 BRONZE (dados brutos do BigQuery)")
print("-"*40)
for name, df in BRONZE_TABLES.items():
    print(f"  {name:30s} {df.count():>8,} rows")

print("\n🟨 SILVER (tratados e validados)")
print("-"*40)
for name, df in SILVER_TABLES.items():
    print(f"  {name:30s} {df.count():>8,} rows")

print("\n🟩 GOLD (produtos analíticos)")
print("-"*40)
for name, df in GOLD_TABLES.items():
    print(f"  {name:30s} {df.count():>8,} rows")

total_bronze = sum(df.count() for df in BRONZE_TABLES.values())
total_silver = sum(df.count() for df in SILVER_TABLES.values())
total_gold = sum(df.count() for df in GOLD_TABLES.values())
print(f"\n  TOTAL: Bronze={total_bronze:,} | Silver={total_silver:,} | Gold={total_gold:,}")

In [0]:
# =============================================================================
# KPI 2 — QUALIDADE DE DADOS (RESUMO)
# =============================================================================

passed = sum(1 for r in quality_results if r.passed)
failed = len(quality_results) - passed
total_checks = len(quality_results)
pct_passed = passed / total_checks * 100

print("\n" + "="*70)
print("  QUALIDADE DE DADOS")
print("="*70)
print(f"\n  Total checks: {total_checks}")
print(f"  ✅ Passou: {passed} ({pct_passed:.0f}%)")
print(f"  ❌ Falhou: {failed}")

if failed > 0:
    print("\n  ⚠️ ALERTAS:")
    for r in quality_results:
        if not r.passed:
            print(f"    ❌ [{r.table_name}] {r.check_name}: {r.message}")
else:
    print("\n  ✅ Todos os checks passaram — pipeline saudável")

# Métricas de freshness
print(f"\n  🕒 Freshness: pipeline executada em {RUN_TIMESTAMP}")
print(f"  💾 Latência estimada: < 2min (extração BQ + transformação Spark)")

In [0]:
# =============================================================================
# KPI 3 — DISTRIBUIÇÃO DA TAXA DE ALFABETIZAÇÃO
# =============================================================================

pdf_ind = gold_indicador_municipio.select("taxa_alfabetizacao", "rede").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma geral
axes[0].hist(pdf_ind["taxa_alfabetizacao"].dropna(), bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(x=80, color="red", linestyle="--", label="Meta referência (80%)")
axes[0].set_xlabel("Taxa de Alfabetização (%)")
axes[0].set_ylabel("Municípios")
axes[0].set_title("Distribuição da Taxa de Alfabetização")
axes[0].legend()

# Box plot por rede
redes = pdf_ind["rede"].unique()
data_by_rede = [pdf_ind[pdf_ind["rede"] == r]["taxa_alfabetizacao"].dropna().values for r in redes]
axes[1].boxplot(data_by_rede, labels=redes, patch_artist=True)
axes[1].set_ylabel("Taxa de Alfabetização (%)")
axes[1].set_title("Distribuição por Rede")
axes[1].axhline(y=80, color="red", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

In [0]:
# =============================================================================
# KPI 4 — PAINEL EXECUTIVO NACIONAL
# =============================================================================

print("\n" + "="*70)
print("  PAINEL EXECUTIVO — VISÃO NACIONAL")
print("="*70)

display(gold_painel_executivo)

# Resumo textual
painel_pdf = gold_painel_executivo.toPandas()
for _, row in painel_pdf.iterrows():
    print(f"\n  Rede: {row['rede']}")
    print(f"    Taxa média nacional: {row['taxa_media_nacional']:.1f}%")
    print(f"    Municípios avaliados: {row['total_municipios']:,}")
    if row.get('meta_2024') and not np.isnan(row['meta_2024']):
        print(f"    Meta 2024: {row['meta_2024']:.1f}% | Gap: {row['gap_nacional']:.1f}pp")
    acima = row.get('municipios_acima_meta', 0)
    pct = row.get('percentual_municipios_acima_meta', 0)
    if acima and not np.isnan(acima):
        print(f"    Municípios acima da meta: {int(acima)} ({pct:.1f}%)")

In [0]:
# =============================================================================
# KPI 5 — EVOLUÇÃO TEMPORAL POR UF (top 5 + bottom 5)
# =============================================================================

pdf_evo = gold_evolucao_temporal.filter(F.col("rede") == "Municipal").select("ano", "sigla_uf", "taxa_alfabetizacao").toPandas()

# Se temos múltiplos anos, plotar evolução
if pdf_evo["ano"].nunique() > 1:
    pivot = pdf_evo.pivot_table(index="ano", columns="sigla_uf", values="taxa_alfabetizacao")
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(ax=ax, legend=False, alpha=0.3, color="gray")
    # Destacar top 5 e bottom 5
    latest = pdf_evo.groupby("sigla_uf")["taxa_alfabetizacao"].mean().sort_values()
    for uf in latest.tail(3).index:
        if uf in pivot.columns:
            pivot[uf].plot(ax=ax, linewidth=2.5, label=f"{uf} (top)")
    for uf in latest.head(3).index:
        if uf in pivot.columns:
            pivot[uf].plot(ax=ax, linewidth=2.5, linestyle="--", label=f"{uf} (bottom)")
    ax.set_title("Evolução da Taxa de Alfabetização por UF (Rede Municipal)")
    ax.set_ylabel("Taxa (%)")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    # Com 1 ano, mostrar ranking
    print("\n=== RANKING UF (Rede Municipal) ===")
    display(gold_evolucao_temporal.filter(F.col("rede") == "Municipal").orderBy(F.col("taxa_alfabetizacao").desc()))

print("\n" + "="*70)
print("  FIM DO DASHBOARD DE MONITORAMENTO")
print("="*70)